In [6]:
"""
XGBoost - 6 modeles : 3 tranches x 2 types (Maison / Appartement).
Routage par commune_prix_m2 (tranche) + is_maison (type).
Cible : residuel_prix_m2 = prix_m2 - commune_prix_m2
"""

import time
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError("pip install xgboost") from exc


def evaluate(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return mae, rmse, r2, mape


print("=" * 70)
print("MODELE : XGBoost 6 sous-modeles (3 tranches x 2 types)")
print("=" * 70)

# ── CHARGEMENT ────────────────────────────────────────────────────────
X_train = pd.read_csv("X_train_optimized.csv")
X_test  = pd.read_csv("X_test_optimized.csv")
y_train = pd.read_csv("y_train_optimized.csv").values.ravel()
y_test  = pd.read_csv("y_test_optimized.csv").values.ravel()

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

# is_maison retire des features (encodee dans la segmentation)
FEATURES = [c for c in X_train.columns if c != "is_maison"]

# ── SEUILS TRANCHES (sur train uniquement) ─────────────────────────────
q33 = round(X_train["commune_prix_m2"].quantile(0.33), -2)  # arrondi a la centaine
q66 = round(X_train["commune_prix_m2"].quantile(0.66), -2)  # arrondi a la centaine

print(f"\nSeuils commune_prix_m2 :")
print(f"  Bas    : < {q33:.0f} euros/m2")
print(f"  Milieu : {q33:.0f} - {q66:.0f} euros/m2")
print(f"  Haut   : > {q66:.0f} euros/m2")


def tranche_label(serie, q33, q66):
    conds = [serie < q33, serie <= q66]
    return np.select(conds, ["bas", "milieu"], default="haut")


tranches_train = tranche_label(X_train["commune_prix_m2"].values, q33, q66)
tranches_test  = tranche_label(X_test["commune_prix_m2"].values,  q33, q66)
types_train    = X_train["is_maison"].values
types_test     = X_test["is_maison"].values

# ── AFFICHAGE TAILLE SEGMENTS ─────────────────────────────────────────
print("\nTaille des 6 segments :")
print(f"{'Segment':<25} {'Train':>10} {'Test':>8}")
print("-" * 45)
for t in ["bas", "milieu", "haut"]:
    for typ, label in [(1, "Maison"), (0, "Appart")]:
        mask_tr = (tranches_train == t) & (types_train == typ)
        mask_te = (tranches_test  == t) & (types_test  == typ)
        print(f"  {t} - {label:<18} {mask_tr.sum():>10,} {mask_te.sum():>8,}")

# ── HYPERPARAMETRES PAR SEGMENT ───────────────────────────────────────
# Haut de gamme : plus d'estimateurs pour capturer la complexite
PARAMS_BASE = dict(
    max_depth=8, learning_rate=0.05, tree_method="hist",
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
N_EST = {"bas": 600, "milieu": 700, "haut": 900}

# ── ENTRAINEMENT ──────────────────────────────────────────────────────
models    = {}
resultats = []
y_pred_global = np.empty(len(y_test))

for tranche in ["bas", "milieu", "haut"]:
    for typ, label in [(1, "Maison"), (0, "Appart")]:
        key = f"{tranche}_{label.lower()}"
        print(f"\n{'─'*55}")
        print(f"Segment : {tranche.upper()} - {label}")

        mask_tr = (tranches_train == tranche) & (types_train == typ)
        mask_te = (tranches_test  == tranche) & (types_test  == typ)

        X_tr = X_train.loc[mask_tr, FEATURES]
        X_te = X_test.loc[mask_te,  FEATURES]
        y_tr = y_train[mask_tr]
        y_te = y_test[mask_te]

        print(f"Train: {len(X_tr):,} | Test: {len(X_te):,}")

        params = {**PARAMS_BASE, "n_estimators": N_EST[tranche]}
        start  = time.time()
        model  = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr)
        t_train = time.time() - start
        print(f"Entraine en {t_train/60:.2f} min")

        y_pred  = model.predict(X_te)
        comm_col = "commune_prix_m2_maison" if typ == 1 else "commune_prix_m2_appart"
        comm     = X_test.loc[mask_te, comm_col].values
        y_pred_clipped = np.clip(y_pred + comm, 300, 15_000)
        mae, rmse, r2, mape = evaluate(y_te + comm, y_pred_clipped)

        print(f"R2   : {r2:.4f}")
        print(f"MAE  : {mae:,.2f} euros/m2  |  MAPE : {mape:.2f} %")

        y_pred_global[mask_te] = y_pred_clipped
        models[key] = model
        resultats.append({
            "segment": key, "tranche": tranche, "type": label,
            "r2": r2, "mae": mae, "rmse": rmse, "mape": mape,
            "n_train": len(X_tr), "train_time_sec": t_train
        })

# ── SCORE GLOBAL ───────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SCORE GLOBAL (6 modeles combines)")
print("=" * 70)

comm_all = np.where(X_test["is_maison"].values == 1, X_test["commune_prix_m2_maison"].values, X_test["commune_prix_m2_appart"].values)
y_true_global = y_test + comm_all
mae_g, rmse_g, r2_g, mape_g = evaluate(y_true_global, y_pred_global)
print(f"R2   : {r2_g:.4f}")
print(f"MAE  : {mae_g:,.2f} euros/m2")
print(f"RMSE : {rmse_g:,.2f} euros/m2")
print(f"MAPE : {mape_g:.2f} %")

resultats.append({
    "segment": "GLOBAL", "tranche": "-", "type": "-",
    "r2": r2_g, "mae": mae_g, "rmse": rmse_g, "mape": mape_g,
    "n_train": len(X_train)
})

# ── TABLEAU RECAPITULATIF ──────────────────────────────────────────────
print("\nTableau recapitulatif :")
df_res = pd.DataFrame(resultats)
print(df_res[["segment", "r2", "mae", "mape"]].to_string(index=False))

# ── SAUVEGARDE ─────────────────────────────────────────────────────────
for key, model in models.items():
    with open(f"model_xgb_{key}.pkl", "wb") as f:
        pickle.dump(model, f)

meta = {"q33": q33, "q66": q66, "features": FEATURES, "prix_m2_min": 300, "prix_m2_max": 15_000}
with open("meta_6modeles.pkl", "wb") as f:
    pickle.dump(meta, f)

df_res.to_csv("result_xgboost_6modeles.csv", index=False)

print("\nFichiers sauvegardes :")
for key in models:
    print(f"  model_xgb_{key}.pkl")
print("  meta_6modeles.pkl  (q33, q66, features — pour la Streamlit)")
print("  result_xgboost_6modeles.csv")


MODELE : XGBoost 6 sous-modeles (3 tranches x 2 types)
X_train : (3581864, 31)
X_test  : (895467, 31)

Seuils commune_prix_m2 :
  Bas    : < 1800 euros/m2
  Milieu : 1800 - 3100 euros/m2
  Haut   : > 3100 euros/m2

Taille des 6 segments :
Segment                        Train     Test
---------------------------------------------
  bas - Maison                938,077  233,540
  bas - Appart                229,311   57,214
  milieu - Maison                682,510  171,518
  milieu - Appart                494,713  123,531
  haut - Maison                371,605   93,135
  haut - Appart                865,648  216,529

───────────────────────────────────────────────────────
Segment : BAS - Maison
Train: 938,077 | Test: 233,540
Entraine en 1.79 min
R2   : 0.3669
MAE  : 406.94 euros/m2  |  MAPE : 38.03 %

───────────────────────────────────────────────────────
Segment : BAS - Appart
Train: 229,311 | Test: 57,214
Entraine en 0.53 min
R2   : 0.3790
MAE  : 366.22 euros/m2  |  MAPE : 31.72 %

───